# Set up

- Install on Linux, macOS, or **WSL2 on Windows** (see [Windows](#windows-linuxmacos-only-tools-require-wsl)). 

## Windows: Linux/macOS-only tools require WSL

- **You need WSL to run this pipeline on Windows.** 
  - Native Windows can browse the code, read `tool_results/`, and 
  - maybe install the Python-only deps via pixi. 
  - It cannot run variant calling, annotation, trees, or the full pipeline.

![Starting WSL from PowerShell](fig/1.png)

- `automatix` / `create_binary_tables` create extra conda environments on demand for tools such as `snippy`, `prokka`, `cd-hit`, `panaroo`, `mashtree`, `bakta`, `pyseer`, and `panacota` (see [sr_amr/envs](sr_amr/envs)). 

- Those packages and their dependencies (e.g. `bcftools`, `aragorn`) are **only published for Linux/macOS on bioconda and have no `win-64` build**. Environment creation on native Windows fails with errors like `nothing provides bcftools` or `PackagesNotFoundError`.

To run ALPAR on a Windows machine, use **WSL2 (Windows Subsystem for Linux)**.

**Where to run each command:** 
  - PowerShell is only for installing WSL/Ubuntu. 
  - The Pixi installer (`curl ... | sh`), `pixi install`, and
  - `alpar` must run in an **Ubuntu** terminal, not PowerShell and not a Cursor PowerShell terminal. That `curl` script is a Linux installer.

1. Install WSL2 with a Linux distribution (run in Windows PowerShell as Administrator):

    `````powershell
    wsl --install -d Ubuntu
    `````
    Let it finish. Restart if Windows asks. After reboot, Ubuntu often opens by itself and asks you to create a Linux username and password. Do that first. Confirm a distro is present with `wsl -l -v`.

## Ubuntu terminal

2. Open the `Ubuntu terminal` (you want a prompt like `user@pc:~$`, not `PS C:\...>`):

    | Method | What to do |
    |---|---|
    | Start menu | Search **Ubuntu** and open it |
    | PowerShell | `wsl` or `wsl -d Ubuntu` |
    | `Cursor` | Terminal dropdown → **New Terminal** → pick a **WSL / Ubuntu** profile (not PowerShell) |

    Cursor is optional. Use it only after Ubuntu exists, and only if the terminal is a WSL profile. If the prompt still starts with `PS`, you are in PowerShell — switch profiles.

3. Inside that `Ubuntu terminal`, install `pixi` (or mamba/conda). 
   - **Do not** run this from PowerShell:

`````shell
    curl -fsSL https://pixi.sh/install.sh | sh
    exec $SHELL
`````

4. Clone the project **into the Linux filesystem** and install from there:

`````shell
    (base) billchung@CEPSNYLPENG1754:/mnt/c/Users/bill.chung/.copilot/repos$ git clone https://github.com/kalininalab/ALPAR.git ~/alpar_demo
    cd ~/alpar_demo
`````

- Reading the Ubuntu prompt

A prompt like `billchung@CEPSNYLPENG1754:~$` means Ubuntu is running (not PowerShell):

```
user@computer:folder$
│    │         │
│    │         └─ current folder (`~` is Linux home)
│    └─ Windows computer name
└─ Linux username
```

| Prompt | Meaning |
|---|---|
| `user@pc:~$` | Ubuntu, in Linux home (`/home/user`) |
| `user@pc:/mnt/c/Users/...$` | Ubuntu, but standing on the Windows disk |
| `PS C:\Users\...>` | PowerShell — do not run `curl ... install.sh` here |

`~` is `/home/<linux-user>`. 

That is **not** your Windows user folder. Ubuntu can still *reach* Windows files here:

```
/mnt/c/Users/<windows-user>
```

Examples:

```shell
ls ~
ls /mnt/c/Users/<windows-user>
ls "/mnt/c/Users/<windows-user>/OneDrive - Danaher/Documents/GitHub/ALPAR"
```

- `Cursor` can open a WSL terminal already sitting in the OneDrive path. That is still Ubuntu, so you can install Pixi there, but `cd ~` before you clone or run the pipeline.

- Windows can see Linux files in File Explorer at `\\wsl$\Ubuntu\home\<linux-user>`.

## Windows copy vs Ubuntu copy

- A clone or copy in Ubuntu (`~/ALPAR`) is a **second, separate folder** from the ALPAR that Cursor has open on Windows (for example `C:\Users\...\Documents\GitHub\ALPAR`). 
- They are not the same files.

| | Windows (Cursor / OneDrive) | Ubuntu |
|---|---|---|
| Path | `C:\Users\...\GitHub\ALPAR` | `/home/<user>/ALPAR` (`~/ALPAR`) |
| Seen in Ubuntu as | `/mnt/c/Users/.../GitHub/ALPAR` | `~/ALPAR` |
| Same files? | No — two copies | No |

Edits in one do **not** appear in the other unless you `git push` / `git pull` or copy files yourself.

If you already have the Windows repo, you do not have to clone again. Copying still creates a second copy:

```shell
mkdir -p ~/ALPAR
cp -a "/mnt/c/Users/<you>/OneDrive - Danaher/Documents/GitHub/ALPAR/." ~/ALPAR/
```

Pick one folder to edit day to day so the two copies do not drift.

- Pushing the Ubuntu copy to GitHub
    - The Ubuntu copy can push to GitHub the same way the Windows copy can, as long as it is a git repo with a remote.

- **`git clone` into `~/ALPAR`:** `origin` is already set. After commits: `git push -u origin HEAD`
- **`cp -a` from OneDrive including `.git`:** remotes and history come along. You can push from either folder.
- **No `.git` folder:** it is just files. Clone instead, or `git init` and `git remote add origin <url>` — that is a new repo, not a continuation of the Windows one.

You need GitHub login **inside Ubuntu** (HTTPS prompt, `gh auth login`, or an SSH key in WSL). Windows credentials do not always carry over.

Do not push both copies to the same branch with different un-pulled commits. Pick one folder to commit from, or always `git pull` before `git push`.

## Pixi 

- Installing with pixi

This repository includes a [pixi.toml](pixi.toml) manifest that installs the dependencies listed in [environment.yml](environment.yml) via [pixi](https://pixi.sh/):

`````shell
pixi install
`````

With pixi, there is no separate `conda activate` step:

- Run one-off commands with `pixi run <command>`, e.g. `pixi run alpar --help`. Pixi resolves/activates the environment automatically for that command.
- Or run `pixi shell` once per terminal session to drop into a shell with the environment already active, similar to `conda activate`.

### Corporate TLS / custom CA certificates

If `pixi install` fails with a TLS or certificate error (common behind a company proxy or custom CA), set 
`tls-root-certs` = "system".


![Getting pixi.toml files from window. See danaher gitHub for this project](fig/2.png)

`````shell
    pixi install
    pixi run alpar --help
`````

    Prefer `~/ALPAR` over `/mnt/c/...`. The Windows filesystem under WSL is slower, and OneDrive-synced folders in particular can break long bioinformatics jobs. If you must use the Windows checkout, the path looks like `/mnt/c/Users/<you>/.../ALPAR`.


# Example

## Prepare input file

To start ALPAR (`automatix` or `create_binary_tables`) you need: seqeunes + folder labels + reference to start.

1. Isolate sequences - the `.fna` files, already in `input_folder`
2. A species references one .gbk/.gbff (--refernece)


![Input data located in Window](fig/3.png)

- .gbk and .gbff are teh same GenBank format.  .gbk is the older name.  NCBI now ships .gbff.  (GenBank flat file)
- A GenBank file is sequence plus gene coordinates and names (CDS, gene, product).
- ALPAR/Snippy needs that so each isolate is compared to the same chromosome and mutations can be labeled.


This repo does not include a reference. The notebook path data/references/reference.gbff is a placeholder. CARD and CABBAGE are not substitutes.

Where to get one: NCBI Assembly — same species as your isolates, prefer a complete RefSeq chromosome.

For your current pair (A. baumannii / imipenem), a standard choice is ATCC 17978:

- Open Acinetobacter baumannii ATCC 17978 (GCF_000015425.1).
- Download → include GenBank (.gbff).
- Save it as EDA/output/alpar_n100/reference.gbff (or any path you pass to --reference).

- If You still need GenBank (gbff) checked as well. If clicking GenBank jumps the highlight to JSONL, the site is just focusing the description pane. Scroll the file-type list and confirm GenBank has a checkmark before you download.

Easier: skip that UI and get the .gbff directly.

- Open `GCF_000015425.1_ASM1542v1_genomic.gbff.gz`
- Save the .gz, unzip it (7-Zip or Windows).
- You should get GCF_000015425.1_ASM1542v1_genomic.gbff.

`alpar_demo` is where I have cloned the ALPAR github.

#  create_binary_tables

- This part takes about 5 hours to process the sample input files

## input

```
cp -r /mnt/c/Users/bill.chung/.copilot/repos/alpar/input_folder ~/alpar_run/input_folder
mkdir -p ~/alpar_run/output
cd ~/alpar_demo
pixi run alpar create_binary_tables \
  -i /home/billchung/alpar_run/input_folder \
  -o /home/billchung/alpar_run/output/imipenem \
  --reference /home/billchung/alpar_run/input_folder/imipenem/reference.gbff \
  --threads 4 \
  --ram 4
```

- `--ram 4` tells every Snippy process it may use 4 GB. 
- With `--threads 4`, four genomes can be in Snippy at once, so that stage uses about 16 GB. 
- `Prokka` ignores this flag, and `CD-HIT` does too.


This computer has 16 cores and 32 GB of RAM. --threads 4 --ram 4 is the right pair here: 16 GB for Snippy and about 16 GB left for Ubuntu and the rest of the pipeline. The current run is already using that.

- On AWS, 
  - on `r7i.4xlarge (16 vCPU, 128 GB)` with `--threads 16 --ram 4`, this step will take about 1 ~ 1.5 hours.
  - on `r7i.8xlarge (32 vCPU, 256 GB)` with `--threads 32 --ram 4`, this step will take about 45 min ~ 75 min .

## OUTPUT

- binary_mutation_table.tsv
- binary_mutation_table_with_gene_presence_absence.tsv
- cd-hit
- mutations_annotations.tsv
- random_names.text
- snippy_processed_strains.txt
- strains.txt

Got `Genome proteins is not in the list of strains to be processed message`.
- it means those columns in the final table are empty.

CD-HIT clustered a file named proteins.faa, which is the protein set Prokka extracted from the ATCC 17978 reference. The cluster names look like this:

proteins_YP_001084105_1_______hemolysin_type_calcium_binding_protein

The pipeline treats the text before the first underscore as the strain name. That text is proteins, which is not one of the 87 genomes (HAOF9Z0Q, 5ELMRUZJ, and the rest). Every reference protein is skipped, and the warning is printed once per protein. None of the real strain names appear in cdhit_output.txt.clstr.

binary_mutation_table.tsv and mutations_annotations.tsv come from Snippy and are unaffected. All 87 genomes are listed in snippy_processed_strains.txt.

binary_mutation_table_with_gene_presence_absence.tsv was still written, but its gene columns do not record which genome has which gene. Prokka leaves both PROKKA_*.faa (that genome's proteins) and proteins.faa (the reference proteins) in each output folder, and this step used proteins.faa.


# binary_table_threshold

`````shell
cd ~/alpar_demo
pixi run alpar binary_table_threshold \
  -i /home/billchung/alpar_run/output/imipenem/binary_mutation_table.tsv \
  -o /home/billchung/alpar_run/output/imipenem \
  --threshold_percentage 20
`````

## OUTPUT

`--threshold_percentage` is a percent from 0 to 100. 20 keeps a column only when that mutation is present in more than 20% of the 87 genomes (at least 18 genomes). The result is written to:

`output/imipenem/binary_table_threshold/binary_mutation_table_threshold_20_percent.tsv`


The default is `0.2`, which is `0.2%`, not 20%. With 87 genomes that only drops mutations seen in a single genome. This step only reads the table, so it should finish in a few minutes.

# phylogenetic_tree

````shell
pixi run alpar phylogenetic_tree \
  -i /home/billchung/alpar_run/output/imipenem/strains.txt \
  -o /home/billchung/alpar_run/output/imipenem \
  --random_names_dict /home/billchung/alpar_run/output/imipenem/random_names.txt
`````

## OUTPUT

The command produced one file, phylogeny/phylogenetic_tree.tree. It is a Newick tree from Mashtree: one line that ends with a semicolon.

Each leaf is a genome’s random ID, the same IDs as in random_names.txt and the binary table. For example, BQ430GXU is Aba_IMI_R_128. The number after the colon is the Mash distance to the neighboring branch. 0.00000 means those two genomes look identical at this resolution. The longer branches are about 0.01, roughly a 1% distance.

Groups inside matching parentheses are genomes Mashtree placed closest together. This is a distance tree from genome sketches, not a mutation-by-mutation alignment tree.

# create_phenotype_table

- GWAS requires a phenotype table. Need to Build that first, then run GWAS on the 20% mutation table.

`````shell
pixi run alpar create_phenotype_table \
  -i /home/billchung/alpar_run/input_folder \
  -o /home/billchung/alpar_run/output/imipenem \
  --random_names_dict /home/billchung/alpar_run/output/imipenem/random_names.txt
`````

## OUTPUT
- phenotype_table.tsv

# gwas

`````shell
pixi run alpar gwas \
  -i /home/billchung/alpar_run/output/imipenem/binary_table_threshold/binary_mutation_table_threshold_20.0_percent.tsv \
  -p /home/billchung/alpar_run/output/imipenem/phenotype_table/phenotype_table.tsv \
  -t /home/billchung/alpar_run/output/imipenem/phylogeny/phylogenetic_tree.tree \
  -o /home/billchung/alpar_run/output/imipenem \
  --threads -1
```

# prps

The score file is written to output/imipenem/prps/prps_score.tsv. Each row is one mutation and a score for how spread out the genomes carrying that mutation are on the tree. A higher score means the mutation appears in distant genomes. A lower score means it is confined to closely related genomes.

`--threads` is accepted by the command, but this step scores mutations one at a time on a single core. Use the 20% table. The unfiltered mutation table has many more columns and takes much longer.

`````shell
pixi run alpar prps \
  -i /home/billchung/alpar_run/output/imipenem/binary_table_threshold/binary_mutation_table_threshold_20.0_percent.tsv \
  -t /home/billchung/alpar_run/output/imipenem/phylogeny/phylogenetic_tree.tree \
  -o /home/billchung/alpar_run/output/imipenem
`````

## OUPTUT 

- `alpar_run/output/imipenem/prps`

# ml

`````shell
pixi run alpar ml \
  -i /home/billchung/alpar_run/output/imipenem/binary_table_threshold/binary_mutation_table_threshold_20.0_percent.tsv \
  -p /home/billchung/alpar_run/output/imipenem/phenotype_table/phenotype_table.tsv \
  -a imipenem \
  -o /home/billchung/alpar_run/output/imipenem \
  --prps /home/billchung/alpar_run/output/imipenem/prps/prps_score.tsv \
  --annotation /home/billchung/alpar_run/output/imipenem/mutations_annotations.tsv \
  --feature_importance_analysis \
  --threads 8 \
  --ram 8
`````

--threads 8 means 8 CPU cores. --ram 8 means a memory budget of 8 GB. For the random-forest command from the previous message, neither one changes the run.

The forest is trained on 1 core, and its importance scores are read from that model. The thread count is passed through only for XGBoost, logistic regression, and permutation importance. The memory value is written into parameters.txt. It changes behavior only for XGBoost: if the table file is larger than ram / 100 GB, XGBoost uses a lower-memory parameter search. Your 20% table is about 12 MB, and 8 / 100 GB is 80 MB, so that switch does not happen.

This is different from the earlier Snippy run, where --ram 4 was given to every genome as a 4 GB allowance. Here the table is loaded into memory either way, and the forest does not receive a 8 GB cap.

Leaving both flags off uses the defaults, 1 thread and 4 GB, and the random forest does the same work. Use --threads 8 if you switch to --ml_algorithm xgb or --feature_importance_analysis_strategy permutation_importance.


`````shell
pixi run alpar ml \
  -i /home/billchung/alpar_run/output/imipenem/binary_table_threshold/binary_mutation_table_threshold_20.0_percent.tsv \
  -p /home/billchung/alpar_run/output/imipenem/phenotype_table/phenotype_table.tsv \
  -a imipenem \
  -o /home/billchung/alpar_run/output/imipenem \
  --prps /home/billchung/alpar_run/output/imipenem/prps/prps_score.tsv \
  --annotation /home/billchung/alpar_run/output/imipenem/mutations_annotations.tsv \
  --feature_importance_analysis \
  --save_model \
  --overwrite
`````

`--overwrite` replaces the current ml folder, then writes the same reports plus the model. Copy anything in that folder to Windows first if you want to keep this copy.

The model file will be:

`/home/billchung/alpar_run/output/imipenem/ml/seed_42_testsize_0.2_resampling_holdout_RF_model.sav`

It is a Python pickle of the scikit-learn random forest. Copy it to Windows with:

`````shell
mkdir -p /mnt/c/Users/bill.chung/.copilot/repos/alpar/output_folder/imipenem
cp -r /home/billchung/alpar_run/output/imipenem/ml /mnt/c/Users/bill.chung/.copilot/repos/alpar/output_folder/imipenem/
`````

## OUTPUT

`/home/billchung/alpar_run/output/imipenem/ml`



`alpar automatix` is the one-shot command. It runs the same six steps as calling these in order:

| Step | Command `automatix` replaces |
|---|---|
| 1 | `alpar create_binary_tables` (Snippy + annotation → mutation TSV, `phenotype_table.tsv`, `strains.txt`) |
| 2 | `alpar binary_table_threshold` |
| 3 | `alpar phylogenetic_tree` (Mashtree). Without `--fast`, it also runs `alpar panacota` first. |
| 4 | `alpar gwas` |
| 5 | `alpar prps` |
| 6 | `alpar ml` |

Use either `automatix` **or** the six commands, not both on the same output folder.

`automatix` does **not** replace `alpar prediction` (score new unlabeled genomes) or `alpar structman`.

## ALPAR steps (run in Ubuntu / WSL)

These commands assume the 100-genome folder was created above and that `REFERENCE` is a real `.gbff` / `.gbk`. If `pixi run alpar` is not on PATH in WSL, use `~/ALPAR` after `git pull`.

Prefer the Linux copy (`~/ALPAR`) over `/mnt/c/.../OneDrive...`.

1. **Create binary tables** — `alpar create_binary_tables` — Snippy + annotation → mutation TSV, `phenotype_table.tsv`, `strains.txt`
2. **Threshold** — `alpar binary_table_threshold` — drop rare mutation columns
3. **Phylogenetic tree** — `alpar phylogenetic_tree` — Mashtree (`--fast` style; skip PanACoTA for n=100)
4. **GWAS** — `alpar gwas` — Pyseer
5. **PRPS** — `alpar prps` — phylogeny-confounder scores
6. **ML** — `alpar ml` — train one model for the sampled antibiotic

`alpar automatix` replaces **all six** of those commands (plus `alpar panacota` if you omit `--fast`). Same inputs: isolate `.fna` folders + `--reference`. Do not run both scripts on the same `-o` folder.

`alpar automatix` is the one-shot command. It runs the same six steps as calling these in order:

| Step | Command `automatix` replaces |
|---|---|
| 1 | `alpar create_binary_tables` (Snippy + annotation → mutation TSV, `phenotype_table.tsv`, `strains.txt`) |Snippy is external tool. ALPAR calls for it and use for variant calling.  This steps takes a long time.  see `sr_amr/binary_tables.py`. For sample of 100 isolate, it took ~ 5 hours. (almost all Snippy)|
| 2 | `alpar binary_table_threshold` |  |
| 3 | `alpar phylogenetic_tree` (Mashtree). Without `--fast`, it also runs `alpar panacota` first. |  |
| 4 | `alpar gwas` |  During the initial try, after gwas, the program stopped. It had to do with creating a folder.  That part is now fixed. 1 ~ 3 hours|
| 5 | `alpar prps` | PRPS is now computing.  Rnning PRPS for binary data.  It scores every mutation on the tree.  This is a pure-Python loop over hundreds of thousands of columns |
| 6 | `alpar ml` | roughly 0.5 to 2 hours|

While the programm is running, you can check the status by 
- `ls ~/ALPAR/EDA/output/alpar_n100/alpar_out/snippy | wc -l`

Once the program finishes evaluating all isolates (`snippy_processed_strains.txt` has 87 lines), those per-isolate folders were merged into the mutation tables.
- Because the script does not pass --keep_temp_files, ALPAR then removes snippy/, prokka/, and temp/ to save disk.
The variants are now in: `~/ALPAR/EDA/output/alpar_n100/alpar_out/binary_mutation_table.tsv`

Also there:

- `binary_mutation_table_with_gene_presence_absence.tsv`
- `mutations_annotations.tsv`
- `phenotype_table.tsv`
- `binary_table_threshold/binary_mutation_table_threshold_0.2_percent.tsv` (`step 2 done`)
- `phylogeny/phylogenetic_tree.tree` (`step 3 done`)
- `gwas/` has started or finished a nested folder. prps/ and ml/ are not there yet. The step script is not running right now.

To keep Snippy folders next time, add --keep_temp_files to create_binary_tables. You do not need them for steps 2–6.


## Result of the analysis

- Random-forest scroes on the held-out isolates

`cat ~/ALPAR/EDA/output/alpar_n100/alpar_out/ml/ml/seed_42_testsize_0.2_resampling_holdout_RF_Result`

- Which mutations the model used (sorted by Gini importance):
`column -t -s $'\t' ~/ALPAR/EDA/output/alpar_n100/alpar_out/ml/ml/seed_42_testsize_0.2_resampling_holdout_RF_FIA_gini | head`
- Each line is a variant vs ATCC 17978, e.g. '2379279','T:C','snp' = position 2379279, T→C.
- 
- GWAS plots: `ls ~/ALPAR/EDA/output/alpar_n100/alpar_out/gwas/gwas/graphs`
- Tree : `cat ~/ALPAR/EDA/output/alpar_n100/alpar_out/phylogeny/phylogenetic_tree.tree`